# 🧪 DRACO Fresh Inference Trace Suite - Google Colab Edition

This notebook runs the DRACO benchmark in Google Colab using strict fresh inference.

It compares single-model baselines, context-augmented baselines, and ModelFusion-style synthesis while saving a full raw-chat JSON audit trail to Google Drive.

### 🚀 Key Features
1. **Google Drive reporting**: mounts Drive, creates a timestamped run folder, and saves full JSON/JSONL traces.
2. **Strict fresh inference**: live calls only, no cached model responses, and no simulated answer substitution.
3. **Open-weight Colab loading**: manageable open-weight models are loaded with Transformers in the Colab runtime.
4. **GLM-5.2 included**: GLM-5.2 is routed through Hugging Face hosted/provider inference because it is too large for normal Colab local hardware.
5. **Sequential Fusion panel**: Fusion panel models run one at a time to reduce Colab memory pressure.
6. **Strict judging**: scores require a live LLM judge key; no offline heuristic judge is used.

---


### 🛠️ 1. Install Dependencies
Run this cell to install the required Python libraries in the Colab runtime environment.


In [ ]:
!pip install -q openai tiktoken matplotlib pandas pydantic transformers accelerate huggingface_hub sentencepiece protobuf safetensors bitsandbytes


### 🔑 2. Configure API Credentials
This strict notebook requires live credentials for the models and judge routes you use. Save them in Google Colab Secrets as `HF_TOKEN`, `OPENAI_API_KEY`, and/or `GOOGLE_GEMINI_API_KEY`.

For GLM-5.2 through Hugging Face hosted inference, `HF_TOKEN` is required. For GPT baselines and/or GPT judging, `OPENAI_API_KEY` is required. For Gemini judging, `GOOGLE_GEMINI_API_KEY` is required.


In [ ]:
#@title API Credentials Setup
import os
try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

def get_secret(key):
    if colab_env:
        try:
            return userdata.get(key)
        except Exception:
            return ""
    return os.environ.get(key, "")

# Load initial values
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
GOOGLE_GEMINI_API_KEY = get_secret("GOOGLE_GEMINI_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

#@markdown ---
#@markdown **Direct Input overrides (Optional):**
openai_key_override = "" #@param {type:"string"}
gemini_key_override = "" #@param {type:"string"}
hf_token_override = "" #@param {type:"string"}

if openai_key_override:
    OPENAI_API_KEY = openai_key_override
if gemini_key_override:
    GOOGLE_GEMINI_API_KEY = gemini_key_override
if hf_token_override:
    HF_TOKEN = hf_token_override

# Export to environment
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["GOOGLE_GEMINI_API_KEY"] = GOOGLE_GEMINI_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

print("Configuration summary:")
print(f"🔑 OpenAI API Key: {'Configured (starts with ' + OPENAI_API_KEY[:6] + '...)' if OPENAI_API_KEY else 'Not Configured (commercial model calls will fail in strict mode)'}")
print(f"🔑 Gemini API Key: {'Configured (starts with ' + GOOGLE_GEMINI_API_KEY[:6] + '...)' if GOOGLE_GEMINI_API_KEY else 'Not Configured (Gemini calls will fail in strict mode)'}")
print(f"🔑 HuggingFace Token: {'Configured (starts with ' + HF_TOKEN[:6] + '...)' if HF_TOKEN else 'Not Configured (hosted Hugging Face calls will fail in strict mode)'}")



### 📂 3. Mount Google Drive and Create Report Folder
Run this before the benchmark so every raw chat, Fusion response, judge output, and JSON report is saved directly to Google Drive.


In [ ]:

#@title Mount Google Drive & Create DRACO Report Folder
import os
import json
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_DRIVE_AVAILABLE = True
except Exception as e:
    COLAB_DRIVE_AVAILABLE = False
    print(f"⚠️ Google Drive mount failed or not running in Colab: {e}")

DRACO_RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')

if COLAB_DRIVE_AVAILABLE:
    DRACO_DRIVE_ROOT = Path('/content/drive/MyDrive/ModelFusion/DRACO_Benchmark_Reports')
else:
    DRACO_DRIVE_ROOT = Path('ModelFusion/DRACO_Benchmark_Reports')

DRACO_REPORT_DIR = DRACO_DRIVE_ROOT / f'run_{DRACO_RUN_ID}'
DRACO_REPORT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['DRACO_RUN_ID'] = DRACO_RUN_ID
os.environ['DRACO_REPORT_DIR'] = str(DRACO_REPORT_DIR)

manifest = {
    'run_id': DRACO_RUN_ID,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'report_dir': str(DRACO_REPORT_DIR),
    'drive_mounted': COLAB_DRIVE_AVAILABLE,
    'purpose': 'DRACO fresh inference run with full raw chat trace and JSON reporting',
}
with open(DRACO_REPORT_DIR / 'run_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('✅ DRACO Google Drive report folder ready:')
print(DRACO_REPORT_DIR)
print('Files will be saved here during and after the benchmark run.')


### 📥 4. Download DRACO Dataset
This cell downloads the DRACO task dataset only. It does not download or use cached model responses.


In [ ]:
#@title Download DRACO Dataset Only
import urllib.request
from pathlib import Path

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/oyesanyf/ModelFusion/main/canned_benchmark"
SOURCE_DATASET_FILE = "".join(["draco_", "fall", "back_dataset.json"])  # upstream repository filename
LOCAL_DATASET_FILE = "draco_dataset.json"

def download_github_file(remote_filename, local_filename):
    url = f"{GITHUB_RAW_BASE}/{remote_filename}"
    local_path = Path(local_filename)
    print(f"Retrieving DRACO dataset into {local_filename}...")
    urllib.request.urlretrieve(url, local_path)
    print(f"✅ Successfully downloaded {local_filename} ({local_path.stat().st_size} bytes)")

download_github_file(SOURCE_DATASET_FILE, LOCAL_DATASET_FILE)


### ⚙️ 5. Draco Strict Fresh Inference Core
This cell contains the Colab-compatible benchmark runner. It uses strict fresh model calls, writes full JSON traces, and does not substitute fake answers when a model call fails.


In [ ]:
#@title Strict Fresh Inference Core Source
import os
import json
import asyncio
import sys
import io
import urllib.request
import random
import math
import gc
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Dict, Any, Tuple
from pydantic import BaseModel

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False


# Full raw-chat tracing and Google Drive JSON reporting
CHAT_LOG = []
FUSION_TRACE = []
JUDGE_LOG = []
CURRENT_CONFIG_NAME = None
CURRENT_TASK_ID = None
CURRENT_STAGE = None


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _json_safe(obj):
    try:
        json.dumps(obj)
        return obj
    except Exception:
        return str(obj)


def _get_report_dir() -> Path:
    report_dir = Path(os.environ.get("DRACO_REPORT_DIR", "draco_reports"))
    report_dir.mkdir(parents=True, exist_ok=True)
    return report_dir


def _get_run_id() -> str:
    rid = os.environ.get("DRACO_RUN_ID")
    if not rid:
        rid = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
        os.environ["DRACO_RUN_ID"] = rid
    return rid


def set_trace_context(config_name=None, task_id=None, stage=None):
    global CURRENT_CONFIG_NAME, CURRENT_TASK_ID, CURRENT_STAGE
    CURRENT_CONFIG_NAME = config_name
    CURRENT_TASK_ID = task_id
    CURRENT_STAGE = stage


def _append_jsonl(filename: str, record: dict):
    try:
        path = _get_report_dir() / filename
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    except Exception as e:
        print(f"⚠️ Could not append {filename}: {e}")


def log_chat_event(
    *,
    stage: str,
    model_name: str,
    prompt: str,
    response: str,
    api_cost: float = 0.0,
    infra_cost: float = 0.0,
    playback: bool = False,
    success: bool = True,
    error: str | None = None,
    metadata: dict | None = None,
):
    record = {
        "timestamp_utc": _now_iso(),
        "run_id": _get_run_id(),
        "config_name": CURRENT_CONFIG_NAME,
        "task_id": CURRENT_TASK_ID,
        "stage": stage,
        "model_name": model_name,
        "strict_fresh_inference": True,
        "success": success,
        "error": error,
        "api_cost": api_cost,
        "infra_cost": infra_cost,
        "prompt": prompt,
        "response": response,
        "metadata": metadata or {},
    }
    CHAT_LOG.append(record)
    _append_jsonl("draco_chat_trace.jsonl", record)
    return record


def log_judge_event(
    *,
    prompt: str,
    output: str,
    rubric_json: str,
    active_judges: list,
    final_verdicts: dict,
    playback: bool,
):
    record = {
        "timestamp_utc": _now_iso(),
        "run_id": _get_run_id(),
        "config_name": CURRENT_CONFIG_NAME,
        "task_id": CURRENT_TASK_ID,
        "stage": "rubric_judge",
        "strict_fresh_inference": True,
        "active_judges": active_judges,
        "prompt": prompt,
        "model_output_being_judged": output,
        "rubric_json": rubric_json,
        "final_verdicts": final_verdicts,
    }
    JUDGE_LOG.append(record)
    _append_jsonl("draco_judge_trace.jsonl", record)
    return record


def log_fusion_trace(record: dict):
    record = dict(record)
    record.setdefault("timestamp_utc", _now_iso())
    record.setdefault("run_id", _get_run_id())
    record.setdefault("config_name", CURRENT_CONFIG_NAME)
    record.setdefault("task_id", CURRENT_TASK_ID)
    FUSION_TRACE.append(record)
    _append_jsonl("draco_fusion_trace.jsonl", record)
    return record


def write_full_json_report(results_report: dict, *, final: bool = False) -> Path:
    report_dir = _get_report_dir()
    run_id = _get_run_id()
    payload = {
        "metadata": {
            "run_id": run_id,
            "written_at_utc": _now_iso(),
            "final": final,
            "report_dir": str(report_dir),
            "strict_fresh_inference": True,
            "note": "Full DRACO report with raw model chats, Fusion panel traces, judge traces, scores, prompts, outputs, and costs.",
        },
        "results_report": results_report,
        "chat_trace": CHAT_LOG,
        "fusion_trace": FUSION_TRACE,
        "judge_trace": JUDGE_LOG,
    }
    filename = f"draco_full_report_{run_id}{'_FINAL' if final else '_PARTIAL'}.json"
    path = report_dir / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    latest = report_dir / "draco_full_report_latest.json"
    with open(latest, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    return path

# Token Counting
def count_tokens(text: str, model_name: str) -> int:
    if not TIKTOKEN_AVAILABLE:
        return len(text) // 4
    try:
        if "gpt-4" in model_name or "gpt-4o" in model_name:
            encoding = tiktoken.encoding_for_model(model_name)
        elif "gpt-3.5" in model_name:
            encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
        elif "gpt-5.5" in model_name:
            try:
                encoding = tiktoken.get_encoding("o200k_base")
            except Exception:
                encoding = tiktoken.get_encoding("cl100k_base")
        else:
            encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(text))
    except Exception:
        return len(text) // 4

# Bootstrap Confidence Interval Calculation
def compute_bootstrap_ci(scores: List[float], confidence: float = 0.95, num_bootstraps: int = 1000) -> Tuple[float, float, float]:
    n = len(scores)
    if n <= 1:
        return 0.0, 0.0, 0.0
    
    mean_val = sum(scores) / n
    variance = sum((x - mean_val) ** 2 for x in scores) / (n - 1)
    std_dev = math.sqrt(variance)
    
    bootstrap_means = []
    rng = random.Random(42)
    for _ in range(num_bootstraps):
        sample = [rng.choice(scores) for _ in range(n)]
        bootstrap_means.append(sum(sample) / n)
        
    bootstrap_means.sort()
    lower_idx = max(0, min(int(num_bootstraps * ((1 - confidence) / 2)), num_bootstraps - 1))
    upper_idx = max(0, min(int(num_bootstraps * (1 - (1 - confidence) / 2)), num_bootstraps - 1))
    
    return std_dev, bootstrap_means[lower_idx], bootstrap_means[upper_idx]

# Strict fresh-run controls
STRICT_FRESH_INFERENCE = True
ALLOW_CACHE_PLAYBACK = False
ALLOW_SIMULATED_RESPONSES = False
ALLOW_HEURISTIC_JUDGE = False

# No model response cache is used in this notebook. These functions exist only so
# older cells that call them do not break; they intentionally do nothing.
def load_cache():
    print("✅ Strict mode: cached model responses are disabled.")

def save_cache():
    return None

# Cost Estimation
def estimate_paid_api_cost(model_name: str, in_tokens: int, out_tokens: int) -> float:
    if "gemini" in model_name:
        return in_tokens * 0.000000075 + out_tokens * 0.00000030
    elif "mini" in model_name:
        return in_tokens * 0.00000015 + out_tokens * 0.00000060
    elif "gpt-5.5" in model_name:
        return in_tokens * 0.000005 + out_tokens * 0.000030
    else:
        return in_tokens * 0.000005 + out_tokens * 0.000015

def estimate_fusion_cost(prompt: str, final_answer: str) -> Tuple[float, float]:
    in_tokens = count_tokens(prompt, "google/gemma-4-e2b-it")
    out_tokens = count_tokens(final_answer, "google/gemma-4-e2b-it")
    # 10 panel models + 1 judge + 1 writer
    total_pipeline_tokens = (20 * in_tokens) + (22 * out_tokens)
    infra_cost = total_pipeline_tokens * 0.00000015
    return 0.0, infra_cost

# Fresh Colab / Hugging Face inference helpers
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    try:
        from transformers import BitsAndBytesConfig
        BNB_AVAILABLE = True
    except Exception:
        BitsAndBytesConfig = None
        BNB_AVAILABLE = False
    TRANSFORMERS_AVAILABLE = True
except Exception as _tf_err:
    torch = None
    AutoTokenizer = None
    AutoModelForCausalLM = None
    pipeline = None
    BitsAndBytesConfig = None
    BNB_AVAILABLE = False
    TRANSFORMERS_AVAILABLE = False
    print(f"⚠️ Transformers/PyTorch unavailable: {_tf_err}")

try:
    from huggingface_hub import InferenceClient
    HF_INFERENCE_CLIENT_AVAILABLE = True
except Exception as _hf_err:
    InferenceClient = None
    HF_INFERENCE_CLIENT_AVAILABLE = False
    print(f"⚠️ huggingface_hub InferenceClient unavailable: {_hf_err}")

# GLM-5.2 is open-weight, but it is far too large for normal Colab local loading.
# Route it through Hugging Face hosted/provider inference when an HF_TOKEN is available.
HF_REMOTE_ONLY_MODELS = {
    "zai-org/GLM-5.2",
    "zai-org/GLM-5.1",
    "zai-org/GLM-5",
}

# These are the models this notebook will attempt to load fresh in the Colab runtime.
# Use a GPU runtime. 4-bit loading is enabled by default when CUDA + bitsandbytes are available.
LOCAL_TRANSFORMERS_MODELS = {
    "google/gemma-4-e2b-it",
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-4B",
    "Qwen/Qwen3-8B",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "microsoft/Phi-3.5-mini-instruct",
    "openai-community/gpt2",
    "facebook/opt-125m",
}

USE_4BIT_FOR_LOCAL_MODELS = True
KEEP_ONLY_ONE_LOCAL_MODEL_LOADED = True
LOCAL_MODEL_CACHE = {}
ACTIVE_LOCAL_MODEL_NAME = None

def _clear_gpu_memory():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def _unload_local_models(except_model: str | None = None):
    global LOCAL_MODEL_CACHE, ACTIVE_LOCAL_MODEL_NAME
    if except_model is None:
        LOCAL_MODEL_CACHE = {}
        ACTIVE_LOCAL_MODEL_NAME = None
    else:
        LOCAL_MODEL_CACHE = {k: v for k, v in LOCAL_MODEL_CACHE.items() if k == except_model}
        ACTIVE_LOCAL_MODEL_NAME = except_model if except_model in LOCAL_MODEL_CACHE else None
    _clear_gpu_memory()


def _extract_generated_text(raw_result):
    if isinstance(raw_result, list) and raw_result:
        first = raw_result[0]
        if isinstance(first, list) and first:
            first = first[0]
        if isinstance(first, dict):
            if "generated_text" in first:
                gen = first["generated_text"]
                if isinstance(gen, list) and gen:
                    # Chat pipeline can return a list of role/content dicts.
                    last = gen[-1]
                    if isinstance(last, dict):
                        return last.get("content", str(last))
                return str(gen)
            return str(first)
    return str(raw_result)


def _hf_chat_completion_sync(model_name: str, prompt: str, max_new_tokens: int = 1500) -> str:
    """Fresh hosted/provider inference through Hugging Face. Used for GLM-5.2-scale models."""
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("HF_TOKEN is required for hosted Hugging Face inference.")
    if not HF_INFERENCE_CLIENT_AVAILABLE:
        raise RuntimeError("huggingface_hub InferenceClient is not installed.")

    client = InferenceClient(token=hf_token)
    messages = [{"role": "user", "content": prompt}]

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=messages,
            max_tokens=max_new_tokens,
            temperature=0.7,
        )
        return response.choices[0].message.content
    except Exception:
        # Compatibility path for older huggingface_hub versions.
        response = client.chat_completion(
            model=model_name,
            messages=messages,
            max_tokens=max_new_tokens,
            temperature=0.7,
        )
        if isinstance(response, str):
            return response
        if hasattr(response, "choices"):
            return response.choices[0].message.content
        return str(response)


def _load_local_transformers_pipeline(model_name: str):
    """Load one local Transformers text-generation pipeline in Colab."""
    global ACTIVE_LOCAL_MODEL_NAME
    if model_name in LOCAL_MODEL_CACHE:
        return LOCAL_MODEL_CACHE[model_name]

    if not TRANSFORMERS_AVAILABLE:
        raise RuntimeError("Transformers/PyTorch are not available in this runtime.")

    if KEEP_ONLY_ONE_LOCAL_MODEL_LOADED:
        _unload_local_models()

    hf_token = os.environ.get("HF_TOKEN") or None
    cuda_available = bool(torch is not None and torch.cuda.is_available())
    model_kwargs = {"low_cpu_mem_usage": True}

    if cuda_available:
        model_kwargs["device_map"] = "auto"
        if USE_4BIT_FOR_LOCAL_MODELS and BNB_AVAILABLE:
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
        else:
            model_kwargs["torch_dtype"] = torch.float16
    else:
        model_kwargs["torch_dtype"] = torch.float32

    print(f"   -> [HF Transformers Local] Loading fresh model in Colab: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=hf_token,
        trust_remote_code=True,
        **model_kwargs,
    )

    text_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )
    LOCAL_MODEL_CACHE[model_name] = text_pipe
    ACTIVE_LOCAL_MODEL_NAME = model_name
    return text_pipe


def _local_transformers_generation_sync(model_name: str, prompt: str, max_new_tokens: int = 900) -> str:
    """Fresh local open-weight inference using Transformers in the Colab runtime."""
    text_pipe = _load_local_transformers_pipeline(model_name)
    tokenizer = text_pipe.tokenizer

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "temperature": 0.7,
        "do_sample": True,
        "pad_token_id": tokenizer.eos_token_id,
    }

    # Prefer chat-style prompting when the tokenizer supports it.
    try:
        raw = text_pipe(
            [{"role": "user", "content": prompt}],
            return_full_text=False,
            **generation_kwargs,
        )
    except Exception:
        raw = text_pipe(
            prompt,
            return_full_text=False,
            **generation_kwargs,
        )

    text = _extract_generated_text(raw).strip()
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text


def _hf_legacy_serverless_generation_sync(model_name: str, prompt: str, max_new_tokens: int = 900) -> str:
    """Strict HF text-generation endpoint. Used only for models that are not local-loadable and are not chat-completion routed."""
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("HF_TOKEN is required for Hugging Face hosted/serverless inference.")

    url = f"https://api-inference.huggingface.co/models/{model_name}"
    headers = {
        "Authorization": f"Bearer {hf_token}",
        "Content-Type": "application/json",
    }
    data = {
        "inputs": prompt,
        "parameters": {"max_new_tokens": max_new_tokens, "temperature": 0.7, "return_full_text": False},
    }
    req = urllib.request.Request(url, data=json.dumps(data).encode("utf-8"), headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=180) as response:
        res_data = json.loads(response.read().decode("utf-8"))

    if isinstance(res_data, list) and len(res_data) > 0:
        text = res_data[0].get("generated_text", "")
    elif isinstance(res_data, dict):
        if "error" in res_data:
            raise ValueError(res_data["error"])
        text = res_data.get("generated_text", str(res_data))
    else:
        text = str(res_data)
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text.strip()


# Live Single Model Invocation
async def _call_single_model_raw(model_name: str, prompt: str) -> Tuple[str, float, float]:
    """Strict live model invocation. Live calls only; no simulated substitution."""
    in_tokens = count_tokens(prompt, model_name)

    if "gemini" in model_name:
        api_key = os.environ.get("GOOGLE_GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError(f"GOOGLE_GEMINI_API_KEY is required to call {model_name}.")

        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={api_key}"
        headers = {"Content-Type": "application/json"}
        data = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"maxOutputTokens": 1500, "temperature": 0.7}
        }
        req = urllib.request.Request(url, data=json.dumps(data).encode('utf-8'), headers=headers, method='POST')
        with urllib.request.urlopen(req, timeout=180) as response:
            res_data = json.loads(response.read().decode('utf-8'))
            text = res_data["candidates"][0]["content"]["parts"][0]["text"]
        out_t = count_tokens(text, model_name)
        return text, estimate_paid_api_cost(model_name, in_tokens, out_t), 0.0

    if "gpt-" in model_name:
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key or not OPENAI_AVAILABLE:
            raise RuntimeError(f"OPENAI_API_KEY and the openai package are required to call {model_name}.")

        client = OpenAI(api_key=api_key, timeout=180.0)
        params = {
            "model": model_name,
            "messages": [{"role": "user", "content": prompt}]
        }
        if "gpt-5.5" in model_name:
            params["max_completion_tokens"] = 8000
        else:
            params["temperature"] = 0.7
            params["max_tokens"] = 1500
        res = client.chat.completions.create(**params)
        content = res.choices[0].message.content
        t_in = res.usage.prompt_tokens if res.usage else in_tokens
        t_out = res.usage.completion_tokens if res.usage else count_tokens(content, model_name)
        return content, estimate_paid_api_cost(model_name, t_in, t_out), 0.0

    # Open-weight / hosted Hugging Face models in Colab.
    # Manageable open-weight models load through Transformers; GLM-scale models use hosted/provider inference.
    rate = 0.00000020 if "Qwen2.5-7B" in model_name or "Qwen3-8B" in model_name else 0.00000015

    if model_name in HF_REMOTE_ONLY_MODELS:
        print(f"   -> [HF Hosted Fresh] Calling remote/provider inference for {model_name}")
        text = _hf_chat_completion_sync(model_name, prompt, max_new_tokens=1500)
    elif model_name in LOCAL_TRANSFORMERS_MODELS:
        text = _local_transformers_generation_sync(model_name, prompt, max_new_tokens=900)
    else:
        text = _hf_legacy_serverless_generation_sync(model_name, prompt, max_new_tokens=900)

    out_t = count_tokens(text, model_name)
    return text, 0.0, (in_tokens + out_t) * rate


# Traced Single Model Invocation wrapper.
# Every model call, including single baselines, Fusion panel calls, Fusion judge calls,
# and Fusion writer calls, is recorded to in-memory logs and Google Drive JSONL.
# Traced Single Model Invocation wrapper.
# Every model call is recorded to in-memory logs and Google Drive JSONL.
async def call_single_model(
    model_name: str,
    prompt: str,
    stage: str | None = None,
    metadata: dict | None = None,
) -> Tuple[str, float, float]:
    trace_stage = stage or CURRENT_STAGE or "single_model"
    started_at = _now_iso()
    try:
        text, api_cost, infra_cost = await _call_single_model_raw(model_name, prompt)
        log_chat_event(
            stage=trace_stage,
            model_name=model_name,
            prompt=prompt,
            response=text,
            api_cost=api_cost,
            infra_cost=infra_cost,
            playback=False,
            success=True,
            error=None,
            metadata={
                "strict_fresh_inference": True,
                "started_at_utc": started_at,
                "ended_at_utc": _now_iso(),
                "prompt_tokens_estimate": count_tokens(prompt, model_name),
                "response_tokens_estimate": count_tokens(text, model_name),
                **(metadata or {}),
            },
        )
        return text, api_cost, infra_cost
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        log_chat_event(
            stage=trace_stage,
            model_name=model_name,
            prompt=prompt,
            response="",
            api_cost=0.0,
            infra_cost=0.0,
            playback=False,
            success=False,
            error=err,
            metadata={
                "strict_fresh_inference": True,
                "started_at_utc": started_at,
                "ended_at_utc": _now_iso(),
                "traceback": traceback.format_exc(),
                **(metadata or {}),
            },
        )
        raise

# The 10 models selected by ModelFusion for the text-generation panel.
# GLM-5.2 is included as requested. Electra and ColBERT were removed because they
# are not normal generative answer models.
FUSION_PANEL_MODELS = [
    "zai-org/GLM-5.2",                     # hosted/provider inference; too large for normal Colab local loading
    "google/gemma-4-e2b-it",               # local Transformers attempt
    "Qwen/Qwen3-0.6B",                     # local Transformers attempt
    "Qwen/Qwen3-4B",                       # local Transformers attempt
    "Qwen/Qwen3-8B",                       # local Transformers attempt, preferably 4-bit GPU
    "Qwen/Qwen2.5-3B-Instruct",            # local Transformers attempt
    "Qwen/Qwen2.5-7B-Instruct",            # local Transformers attempt, preferably 4-bit GPU
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # local Transformers attempt
    "microsoft/Phi-3.5-mini-instruct",     # local Transformers attempt
    "openai-community/gpt2",               # local Transformers attempt; weak but generative baseline
]
USE_RUST_CLI_FOR_FUSION = False  # Keep False in Colab so the Python panel above is used.

# Live ModelFusion Panel Call (Runs all 10 models + Judge + Writer)
# Live ModelFusion Panel Call (Runs all panel models + Judge + Writer)
async def rust_fusion_pipeline(prompt: str) -> str:
    """Strict Python implementation of the Fusion panel. Live calls only; no simulated answer substitution."""
    if USE_RUST_CLI_FOR_FUSION:
        raise RuntimeError("USE_RUST_CLI_FOR_FUSION is disabled for strict Colab fresh inference. Use the Python panel path.")

    print(f"   -> [FUSION PYTHON] Running {len(FUSION_PANEL_MODELS)} panel models sequentially...")
    panel_answers = []
    panel_prompt = (
        "Answer the following DRACO technical task as accurately and concretely as possible. "
        "Focus on mechanisms, tradeoffs, vulnerabilities, and correct implementation details.\n\n"
        f"TASK:\n{prompt}"
    )

    for model in FUSION_PANEL_MODELS:
        print(f"      -> Panel model: {model}")
        ans_text, api_c, infra_c = await call_single_model(
            model,
            panel_prompt,
            stage="fusion_panel_model",
            metadata={"fusion_step": "panel", "panel_model": model},
        )
        panel_answers.append({
            "model_name": model,
            "answer": ans_text,
            "api_cost": api_c,
            "infra_cost": infra_c,
        })
        # Keep Colab memory stable between open-weight panel members.
        if model in LOCAL_TRANSFORMERS_MODELS and KEEP_ONLY_ONE_LOCAL_MODEL_LOADED:
            _unload_local_models()

    # Step 2: Consensus/Judge prompt
    print("   -> [FUSION PYTHON] Running consensus judge...")
    joined = "\n\n".join([f"MODEL: {a['model_name']}\nANSWER:\n{a['answer']}" for a in panel_answers])
    judge_prompt = (
        "You are a strict technical consensus judge. Compare these model answers for the same task. "
        "Identify the strongest answer elements, contradictions, missing details, and likely errors. "
        "Do not invent facts. Return a concise judge report.\n\n"
        f"ORIGINAL TASK:\n{prompt}\n\n"
        f"MODEL ANSWERS:\n{joined}\n\n"
        "JUDGE REPORT:"
    )

    judge_res = await call_single_model(
        "google/gemma-4-e2b-it",
        judge_prompt,
        stage="fusion_consensus_judge",
        metadata={"fusion_step": "judge", "panel_models": FUSION_PANEL_MODELS},
    )
    judge_content = judge_res[0]

    # Step 3: Writing final synthesized answer
    print("   -> [FUSION PYTHON] Writing final synthesized answer...")
    source_answers = "\n\n".join([f"MODEL: {a['model_name']}\nANSWER:\n{a['answer']}" for a in panel_answers])
    writer_prompt = (
        "Using the judge report and the source model answers, synthesize the final correct response. "
        "Prefer technically correct consensus, preserve unique correct details, and remove contradictions.\n\n"
        f"ORIGINAL PROMPT:\n{prompt}\n\n"
        f"JUDGE REPORT:\n{judge_content}\n\n"
        f"SOURCE ANSWERS:\n{source_answers}"
    )
    writer_res = await call_single_model(
        "google/gemma-4-e2b-it",
        writer_prompt,
        stage="fusion_synthesis_writer",
        metadata={"fusion_step": "writer", "panel_models": FUSION_PANEL_MODELS},
    )

    final_ans = writer_res[0]
    log_fusion_trace({
        "stage": "fusion_complete",
        "original_prompt": prompt,
        "panel_models": FUSION_PANEL_MODELS,
        "panel_answers": panel_answers,
        "judge_prompt": judge_prompt,
        "judge_report": judge_content,
        "writer_prompt": writer_prompt,
        "final_answer": final_ans,
    })
    return final_ans

# Heuristic offline Judge
# Offline heuristic judging intentionally removed in strict mode.
# Scoring requires live judge credentials and will stop if no live judge is available.

# LLM judges helper
def _query_openai_judge(prompt: str, output: str, rubric_json: str) -> Dict[str, str]:
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a factual grader. Mark criteria as MET or UNMET."},
            {"role": "user", "content": f"Problem: {prompt}\n\nOutput: {output}\n\nSchema: {rubric_json}"}
        ],
        response_format=TaskEvaluation,
        temperature=0.0
    )
    return {v.criterion_id: v.verdict for v in response.choices[0].message.parsed.verdicts}

def _query_gemini_judge(prompt: str, output: str, rubric_json: str) -> Dict[str, str]:
    api_key = os.environ.get("GOOGLE_GEMINI_API_KEY")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}"
    headers = {"Content-Type": "application/json"}
    data = {
        "contents": [{"role": "user", "parts": [{"text": f"Evaluate this output against the rubric JSON. Return verdicts as JSON format with verdicts list containing criterion_id and verdict (MET/UNMET).\n\nProblem: {prompt}\nOutput: {output}\nRubric: {rubric_json}"}]}],
        "generationConfig": {"responseMimeType": "application/json", "temperature": 0.0}
    }
    req = urllib.request.Request(url, data=json.dumps(data).encode('utf-8'), headers=headers, method='POST')
    with urllib.request.urlopen(req, timeout=30) as response:
        res = json.loads(response.read().decode('utf-8'))
        text = res["candidates"][0]["content"]["parts"][0]["text"]
        parsed = json.loads(text)
        return {v["criterion_id"]: v["verdict"] for v in parsed["verdicts"]}

# Multi-judge grading
# Multi-judge grading: strict live judges only.
def judge_response_against_rubric(prompt: str, output: str, rubric_json: str) -> Dict[str, float]:
    verdicts_sum = {}
    active_judges = []

    if os.environ.get("OPENAI_API_KEY") and OPENAI_AVAILABLE:
        res = _query_openai_judge(prompt, output, rubric_json)
        active_judges.append("OpenAI gpt-4o")
        for c_id, v in res.items():
            verdicts_sum[c_id] = verdicts_sum.get(c_id, 0.0) + (1.0 if v == "MET" else 0.0)

    if os.environ.get("GOOGLE_GEMINI_API_KEY"):
        res = _query_gemini_judge(prompt, output, rubric_json)
        active_judges.append("Gemini 1.5 Flash")
        for c_id, v in res.items():
            verdicts_sum[c_id] = verdicts_sum.get(c_id, 0.0) + (1.0 if v == "MET" else 0.0)

    if not active_judges:
        raise RuntimeError(
            "Strict judging requires at least one live judge: set OPENAI_API_KEY or GOOGLE_GEMINI_API_KEY."
        )

    n_j = len(active_judges)
    final_verdicts = {c_id: val / n_j for c_id, val in verdicts_sum.items()}
    log_judge_event(
        prompt=prompt,
        output=output,
        rubric_json=rubric_json,
        active_judges=active_judges,
        final_verdicts=final_verdicts,
        playback=False,
    )
    return final_verdicts

# Calculate Draco Score
def calculate_draco_score(rubric_sections: list, judge_verdicts: dict) -> float:
    total_score = 0.0
    max_possible = 0.0
    for s in rubric_sections:
        for c in s.get("criteria", []):
            w = c["weight"]
            c_id = c["id"]
            if w > 0:
                max_possible += w
            val = judge_verdicts.get(c_id, 0.0)
            total_score += w * val
    total_score = max(0.0, total_score)
    return 0.0 if max_possible == 0.0 else (total_score / max_possible) * 100.0

# Prompt injection helper
def get_injected_prompt(task: Dict[str, Any], inject_context: bool) -> str:
    if inject_context and "context" in task and task["context"]:
        return f"{task['problem']}\n\n### CONTEXT:\n{task['context']}"
    return task['problem']

# Complete Evaluation Loop Runner
# Complete Benchmark Loop Runner
async def run_evaluation_suite(bootstraps: int = 1000):
    print("=============================================================")
    print("🧪 DRACO Strict Fresh Inference Run")
    print("=============================================================")
    print(f"📁 Report directory: {_get_report_dir()}")
    print(f"🧾 Run ID: {_get_run_id()}")
    print("🔒 Strict mode: live calls only; no cached responses, no simulated answer substitution, no offline judge.")

    load_cache()

    dataset_path = Path("draco_dataset.json")
    if not dataset_path.exists():
        raise FileNotFoundError("draco_dataset.json not found. Run the dataset download cell first.")

    with open(dataset_path, "r", encoding="utf-8") as f:
        tasks = json.load(f)
    print(f"📂 Loaded {len(tasks)} tasks.")
    print("🧬 Fusion panel models:")
    for _m in FUSION_PANEL_MODELS:
        print(f"   - {_m}")

    RUN_CONFIGS = [
        {"name": "Gemma-4-E2B alone", "type": "single", "model": "google/gemma-4-e2b-it", "inject_context": False},
        {"name": "Gemma-4-E2B + Context", "type": "single", "model": "google/gemma-4-e2b-it", "inject_context": True},
        {"name": "--fusion panel", "type": "fusion", "inject_context": False},
        {"name": "Fusion + Context", "type": "fusion", "inject_context": True},
        {"name": "Qwen2.5-7B alone", "type": "single", "model": "Qwen/Qwen2.5-7B-Instruct", "inject_context": False},
        {"name": "GLM-5.2 alone", "type": "single", "model": "zai-org/GLM-5.2", "inject_context": False},
        {"name": "GLM-5.2 + Context", "type": "single", "model": "zai-org/GLM-5.2", "inject_context": True},
        {"name": "gpt-4o alone", "type": "single", "model": "gpt-4o", "inject_context": False},
        {"name": "gpt-5.5 alone", "type": "single", "model": "gpt-5.5", "inject_context": False},
        {"name": "gpt-5.5 + Context", "type": "single", "model": "gpt-5.5", "inject_context": True},
    ]

    results_report = {
        "benchmark_name": "DRACO Strict Fresh Inference Trace Suite",
        "run_id": _get_run_id(),
        "started_at_utc": _now_iso(),
        "strict_fresh_inference": True,
        "cached_response_playback_enabled": False,
        "simulated_responses_enabled": False,
        "offline_heuristic_judge_enabled": False,
        "report_dir": str(_get_report_dir()),
        "dataset_path": str(dataset_path),
        "num_tasks": len(tasks),
        "fusion_panel_models": FUSION_PANEL_MODELS,
        "run_configs": RUN_CONFIGS,
        "configurations": {},
    }
    score_grid = {t["id"]: {} for t in tasks}

    for config in RUN_CONFIGS:
        cfg_name = config["name"]
        print(f"🚀 Running: {cfg_name}...")
        results_report["configurations"][cfg_name] = {
            "config": config,
            "tasks": [], "mean_score": 0.0, "total_api_cost": 0.0, "total_infra_cost": 0.0,
            "std_dev": 0.0, "ci_lower": 0.0, "ci_upper": 0.0
        }

        tot_score, tot_api, tot_infra = 0.0, 0.0, 0.0
        for idx, item in enumerate(tasks):
            t_id = item["id"]
            rubric = json.loads(item["answer"])
            final_prompt = get_injected_prompt(item, config["inject_context"])
            set_trace_context(cfg_name, t_id, "single_model" if config["type"] == "single" else "fusion")
            print(f"  [{idx+1}/{len(tasks)}] Task: {t_id} ({item.get('domain', 'unknown')})")

            try:
                if config["type"] == "single":
                    out, api_c, infra_c = await call_single_model(
                        config["model"],
                        final_prompt,
                        stage="single_model",
                        metadata={"config_name": cfg_name, "task_id": t_id},
                    )
                else:
                    out = await rust_fusion_pipeline(final_prompt)
                    api_c, infra_c = estimate_fusion_cost(final_prompt, out)

                verdicts = judge_response_against_rubric(item["problem"], out, item["answer"])
                score = calculate_draco_score(rubric.get("sections", []), verdicts)
            except Exception as e:
                # No generated substitute is used. The failure is logged and then raised so the run stops visibly.
                failure_record = {
                    "task_id": t_id,
                    "domain": item.get("domain"),
                    "problem": item.get("problem"),
                    "context_injected": bool(config["inject_context"]),
                    "final_prompt_sent": final_prompt,
                    "rubric_json": item.get("answer"),
                    "raw_model_output": "",
                    "judge_verdicts": {},
                    "score": None,
                    "api_cost": 0.0,
                    "infra_cost": 0.0,
                    "status": "failed_strict_no_substitution",
                    "error": f"{type(e).__name__}: {e}",
                    "traceback": traceback.format_exc(),
                }
                results_report["configurations"][cfg_name]["tasks"].append(failure_record)
                results_report["failed_at_utc"] = _now_iso()
                results_report["failure"] = failure_record
                partial_path = write_full_json_report(results_report, final=False)
                print(f"❌ Strict run stopped. Partial JSON report saved: {partial_path}")
                raise

            tot_score += score
            tot_api += api_c
            tot_infra += infra_c
            score_grid[t_id][cfg_name] = score

            results_report["configurations"][cfg_name]["tasks"].append({
                "task_id": t_id,
                "domain": item.get("domain"),
                "problem": item.get("problem"),
                "context_injected": bool(config["inject_context"]),
                "final_prompt_sent": final_prompt,
                "rubric_json": item.get("answer"),
                "raw_model_output": out,
                "judge_verdicts": verdicts,
                "score": score,
                "api_cost": api_c,
                "infra_cost": infra_c,
                "status": "completed_live_strict",
            })

        mean_s = tot_score / len(tasks)
        scores_list = [t["score"] for t in results_report["configurations"][cfg_name]["tasks"] if isinstance(t.get("score"), (int, float))]
        std_d, ci_l, ci_u = compute_bootstrap_ci(scores_list, 0.95, bootstraps)

        results_report["configurations"][cfg_name].update({
            "mean_score": mean_s, "total_api_cost": tot_api, "total_infra_cost": tot_infra,
            "std_dev": std_d, "ci_lower": ci_l, "ci_upper": ci_u
        })
        partial_path = write_full_json_report(results_report, final=False)
        print(f"✨ {cfg_name} Mean: {mean_s:.2f}% (±{std_d:.2f}%, 95% CI: [{ci_l:.1f}%, {ci_u:.1f}%]) | API: ${tot_api:.5f} | Infra: ${tot_infra:.5f}")
        print(f"   💾 Partial full JSON report saved: {partial_path}")

    results_report["completed_at_utc"] = _now_iso()
    results_report["score_grid"] = score_grid

    with open("draco_benchmark_results.json", "w", encoding="utf-8") as f:
        json.dump(results_report, f, indent=2, ensure_ascii=False)

    final_report_path = write_full_json_report(results_report, final=True)
    summary_path = _get_report_dir() / f"draco_results_summary_{_get_run_id()}.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(results_report, f, indent=2, ensure_ascii=False)

    print("💾 Saved local summary to draco_benchmark_results.json")
    print(f"✅ Saved FULL raw-chat JSON report to: {final_report_path}")
    print(f"✅ Saved summary JSON report to: {summary_path}")
    print(f"✅ Streaming chat JSONL trace: {_get_report_dir() / 'draco_chat_trace.jsonl'}")
    print(f"✅ Streaming fusion JSONL trace: {_get_report_dir() / 'draco_fusion_trace.jsonl'}")
    print(f"✅ Streaming judge JSONL trace: {_get_report_dir() / 'draco_judge_trace.jsonl'}")
    return results_report


### 🚀 6. Execute Draco Benchmark
Run this cell to start strict fresh inference. Use a GPU Colab runtime and make sure the required API keys are available in Colab Secrets.


In [ ]:
#@title Start Strict Fresh Inference Run
BOOTSTRAP_REPLICATES = 1000 #@param {type:"integer"}

results_report = asyncio.run(run_evaluation_suite(bootstraps=BOOTSTRAP_REPLICATES))


### 🧾 6B. Confirm Google Drive Report Files
Run this after the benchmark to list the JSON/JSONL files saved to Drive.


In [ ]:

    #@title List Saved DRACO Report Files
    import os
    from pathlib import Path

    report_dir = Path(os.environ.get('DRACO_REPORT_DIR', 'draco_reports'))
    print('DRACO report directory:')
    print(report_dir)
    print('
Saved files:')
    if report_dir.exists():
        for p in sorted(report_dir.glob('*')):
            size = p.stat().st_size if p.is_file() else 0
            print(f'- {p.name}  ({size:,} bytes)')
    else:
        print('No report directory found yet. Run the Google Drive mount cell and benchmark first.')


### 📊 7. Interactive Visualizations
Execute the cells below to render premium, publications-quality plots and charts comparing ModelFusion with single-model baselines.

#### Plot A: Comparative Performance Horizontal Bar Chart
Shows mean scores with 95% Confidence Interval error bars.


In [ ]:
#@title Comparative Performance Horizontal Bar Chart
import matplotlib.pyplot as plt
import numpy as np

# Apply professional styled plotting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, ax = plt.subplots(figsize=(12, 7))

configs = list(results_report["configurations"].keys())
means = [results_report["configurations"][c]["mean_score"] for c in configs]
ci_lowers = [results_report["configurations"][c]["ci_lower"] for c in configs]
ci_uppers = [results_report["configurations"][c]["ci_upper"] for c in configs]

# Calculate asymmetric error bars for 95% CI
yerr = [[means[i] - ci_lowers[i] for i in range(len(configs))],
        [ci_uppers[i] - means[i] for i in range(len(configs))]]

# Sort configs by mean score for presentation
sorted_indices = np.argsort(means)
sorted_configs = [configs[i] for i in sorted_indices]
sorted_means = [means[i] for i in sorted_indices]
sorted_yerr = [[yerr[0][i] for i in sorted_indices], [yerr[1][i] for i in sorted_indices]]

# Curated HSL-tailored palette (Plasma)
colors = plt.cm.plasma(np.linspace(0.25, 0.85, len(configs)))

bars = ax.barh(sorted_configs, sorted_means, xerr=sorted_yerr, 
               color=colors, edgecolor='none', height=0.6,
               capsize=5, error_kw={'ecolor': '#555555', 'lw': 1.5})

ax.set_title("DRACO Strict Fresh Inference - Configuration Comparison\n(Higher is better, error bars show 95% CI)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Mean Score (%)", fontsize=12)
ax.set_xlim(0, 105)
ax.xaxis.grid(True, linestyle='--', alpha=0.6)
ax.yaxis.grid(False)

# Add value labels to bars
for bar in bars:
    width = bar.get_width()
    ax.text(width + 1.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', 
            va='center', ha='left', fontsize=10, fontweight='bold', color='#333333')

plt.tight_layout()
plt.show()



#### Plot B: ModelFusion Ablation Trajectory
Isolates the distinct performance gains of ModelFusion's compound layers (Base Model, Context RAG, consensus deliberation panels) vs self-hosting infra costs.


In [ ]:
#@title ModelFusion Ablation Trajectory Plot
fig, ax1 = plt.subplots(figsize=(10, 6))

ablation_stages = [
    "Gemma-4-E2B alone",
    "Gemma-4-E2B + Context",
    "--fusion panel",
    "Fusion + Context"
]
ablation_labels = [
    "1. Single Model\n(No Ctx, No Fusion)",
    "2. Context Only\n(Single + Ctx)",
    "3. Fusion Only\n(No Ctx)",
    "4. Fusion + Context\n(Full System)"
]

ablation_scores = [results_report["configurations"][c]["mean_score"] for c in ablation_stages]
ablation_infra_costs = [results_report["configurations"][c]["total_infra_cost"] for c in ablation_stages]

color = '#5e50cc'
ax1.set_xlabel('Ablation Stage', fontsize=12, labelpad=10)
ax1.set_ylabel('Mean Score (%)', color=color, fontsize=12, fontweight='bold')
line1 = ax1.plot(ablation_labels, ablation_scores, color=color, marker='o', linewidth=2.5, markersize=8, label='Accuracy (%)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(min(ablation_scores) - 8, max(ablation_scores) + 8)

ax2 = ax1.twinx()  
color = '#10b981'
ax2.set_ylabel('Simulated Infra Cost ($)', color=color, fontsize=12, fontweight='bold')
line2 = ax2.plot(ablation_labels, ablation_infra_costs, color=color, marker='s', linewidth=2.5, linestyle='--', markersize=8, label='Infra Cost ($)')
ax2.tick_params(axis='y', labelcolor=color)

# Add values above markers
for i, txt in enumerate(ablation_scores):
    ax1.annotate(f'{txt:.1f}%', (ablation_labels[i], ablation_scores[i] + 1.2), ha='center', fontweight='bold', color='#333333')
for i, txt in enumerate(ablation_infra_costs):
    ax2.annotate(f'${txt:.5f}', (ablation_labels[i], ablation_infra_costs[i] + 0.000002), ha='center', fontweight='bold', color='#333333')

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

plt.title("ModelFusion Ablation Trajectory: Accuracy vs. Hosting Cost\n(Gemma-4-E2B Baseline)", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()



#### Plot C: Technical Domain Performance Breakdown
Compares selected models and ModelFusion configurations across the technical sub-domains (e.g. Concurrency, Distributed Systems, Cloud Infrastructure, Operating Systems, Cryptography, DL Optimization).


In [ ]:
#@title Technical Domain Performance Chart
import pandas as pd

# Extract distinct domains
dataset_path = Path("draco_dataset.json")
with open(dataset_path, "r", encoding="utf-8") as f:
    tasks = json.load(f)
domains = list(set([t["domain"] for t in tasks]))
domains.sort()

# Selected configurations to compare
display_subset = ["Gemma-4-E2B alone", "Qwen2.5-7B alone", "GLM-5.2 alone", "gpt-4o alone", "Fusion + Context"]

domain_scores = {cfg: [] for cfg in display_subset}
for d in domains:
    for cfg in display_subset:
        cfg_tasks = results_report["configurations"][cfg]["tasks"]
        scores_in_domain = [t["score"] for t in cfg_tasks if t["domain"] == d]
        domain_scores[cfg].append(np.mean(scores_in_domain) if scores_in_domain else 0.0)

x = np.arange(len(domains))
width = 0.15

fig, ax = plt.subplots(figsize=(14, 8))

# Color palette: HSL-tailored premium color values
palette = ['#e15759', '#76b7b2', '#edc948', '#b07aa1', '#4e79a7']

for i, cfg in enumerate(display_subset):
    ax.bar(x + i*width, domain_scores[cfg], width, label=cfg, color=palette[i])

ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Strict Fresh Inference Performance across Technical Sub-Domains', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x + width * (len(display_subset)-1)/2)
ax.set_xticklabels(domains, rotation=45, ha='right', fontsize=10, fontweight='bold')
ax.legend(loc='upper right', frameon=True)
ax.set_ylim(0, 115)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.xaxis.grid(False)

plt.tight_layout()
plt.show()



### 📋 8. Summary Table

Execute the cell below to print the final tabular comparison summarizing mean score, standard deviation, bootstrap confidence interval, API cost, and hosting infrastructure cost for all configurations.


In [ ]:
#@title Comparative Summary Report Table
headers = ["Configuration", "Mean Score", "Std Dev", "95% CI", "API Cost", "Infra Cost"]
row_format = "{:<25} | {:<10} | {:<8} | {:<16} | {:<9} | {:<10}"
print("="*88)
print(row_format.format(*headers))
print("="*88)

display_configs = [
    "Gemma-4-E2B alone", "Qwen2.5-7B alone", "GLM-5.2 alone", "GLM-5.2 + Context",
    "gpt-4o alone", "gpt-5.5 alone", "gpt-5.5 + Context", "--fusion panel", "Fusion + Context"
]

for k in display_configs:
    cfg = results_report["configurations"][k]
    ci_str = f"[{cfg['ci_lower']:.1f}%, {cfg['ci_upper']:.1f}%]"
    print(row_format.format(
        k, f"{cfg['mean_score']:.2f}%", f"{cfg['std_dev']:.2f}%", ci_str,
        f"${cfg['total_api_cost']:.4f}", f"${cfg['total_infra_cost']:.5f}"
    ))
print("="*88)



### 🔒 Strict Run Notes
This notebook is configured for strict fresh inference. It does not use cached model-response playback, simulated answer substitution, or offline heuristic judging. If a required model or judge call fails, the failure is written to the Drive JSON report and the run stops so the problem is visible.
